# 00 — Start here: question, contract, and offline workspace

**Estimated time:** 20 minutes<br>
**Prerequisites:** none<br>
**Learner-produced evidence:** a ready/not-ready asset table and a written learning goal

## Learning objectives

- Understand the experiment question and structured-output task.
- Verify that the local study assets are present without using a network.
- Distinguish a learning result from a production-readiness claim.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## The experiment question

Can a small LoRA adapter improve structured customer-support intent
prediction over the strongest meaningful baseline while preserving
strict JSON, known labels, and safe response wording?

The evidence lifecycle is **baseline → change → result → decision**.
Training success alone is not a decision.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

## Check this prepared machine

The paths below stay local. A red row means preparation is incomplete;
it does not trigger a download. The model, source archive, generated
data, adapters, and MLflow database are deliberately ignored by Git.


In [ ]:
from aai_local_finetuning.offline import (
    apple_silicon_status,
    asset_checks,
    deny_network,
    prove_socket_denial,
)
from aai_local_finetuning.settings import load_settings

settings = load_settings()
machine = apple_silicon_status()
readiness = [
    {
        "asset": check.name,
        "ready": check.ready,
        "detail": check.detail,
    }
    for check in [machine, *asset_checks(settings)]
]
readiness

## Prove the Python guard

This scoped check deliberately denies Python socket connections. The
notebook also enables the supported offline flags before importing
model or tracking libraries. Turning Wi-Fi off once before departure
remains the strongest rehearsal because native libraries are outside
Python's complete control.


In [ ]:
with deny_network():
    prove_socket_denial()
"Python socket guard passed"

## The output boundary

The model must return exactly four typed fields. A plausible-looking
sentence is not enough, and a high intent score cannot conceal invalid
or unsafe generated output.


In [ ]:
from aai_local_finetuning.evaluation import SupportOutput

SupportOutput.model_json_schema()

## Exercise — write your evidence question

Replace the default sentence with the question you want the final
decision to answer. Success means it names a baseline, a change, a
frozen evaluation boundary, and at least one output-quality gate.


In [ ]:
my_evidence_question = (
    "Does the LoRA change beat the strongest baseline on frozen macro-F1 "
    "while meeting strict schema and response-policy gates?"
)
assert all(
    term in my_evidence_question.lower()
    for term in ("lora", "baseline", "frozen", "schema")
)
my_evidence_question

**Hint:** describe the comparison and its evidence, not the result you
hope to see. Keep production suitability outside this learning claim.


## Checkpoint

You can now explain why offline readiness, strict output validation,
and a frozen comparison are three different concerns.

**Next:** `01_dataset_provenance_and_license.ipynb` examines whether the
source may be used for this curriculum and what remains unproven.
